# H2-1 확장 — 7단계: 다중비교 보정 (Benjamini-Hochberg)

**6단계 결과 요약** (raw p-value 기준)
- 유의(p<0.05): TypeC×고지출(coef=-6.25, p=0.0004), TypeB×고지출(coef=-2.43, p=0.0118)
- 나머지 7개 조합: 유의하지 않음 (TypeC×저지출은 계수는 크지만 n=28, p=0.163으로 노이즈 가능성)

**이번 단계**: Type×계층 9개 조합을 한꺼번에 검정했으므로, 개별 p-value를 그대로 "유의하다"고
판단하면 다중비교 문제(false positive 확률 증가)가 생김. Benjamini-Hochberg(FDR) 보정을 적용해
"9개 중 몇 개가 보정 후에도 살아남는지" 확정.

**예상(6단계 해석 시 암산)**: TypeC×고지출만 살아남고, TypeB×고지출은 근소하게 탈락할 가능성.
이번 단계에서 정확히 계산해 확정.


## 0. 환경 설정

In [1]:
# 최초 1회만 실행
!pip install linearmodels statsmodels --break-system-packages -q


In [2]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
from statsmodels.stats.multitest import multipletests

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

DATA_DIR = "data/"

TIER_MIN_WEEK, TIER_MAX_WEEK = 17, 32
CAMP_MIN_WEEK, CAMP_MAX_WEEK = 33, 101
CELL_SIZE_THRESH = 30


## 1~6단계 파이프라인 재구성 (동일 로직 — 새 내용 없음)

In [3]:
tx = pd.read_csv(DATA_DIR + "transaction_data.csv",
                  usecols=["household_key", "DAY", "WEEK_NO", "SALES_VALUE"])
campaign_table = pd.read_csv(DATA_DIR + "campaign_table.csv")
campaign_desc  = pd.read_csv(DATA_DIR + "campaign_desc.csv")

all_households = tx["household_key"].unique()

# --- 계층 고정 (17~32주) ---
tier_window = tx[(tx["WEEK_NO"] >= TIER_MIN_WEEK) & (tx["WEEK_NO"] <= TIER_MAX_WEEK)]
n_weeks_tier = TIER_MAX_WEEK - TIER_MIN_WEEK + 1
avg_weekly_spend = (tier_window.groupby("household_key")["SALES_VALUE"].sum() / n_weeks_tier).reindex(all_households).fillna(0)
tier3 = pd.qcut(avg_weekly_spend, 3, labels=["저지출", "중지출", "고지출"])
tier_df = pd.DataFrame({"avg_weekly_spend_pre": avg_weekly_spend, "tier3": tier3})
tier_df["was_zero_pre"] = (tier_df["avg_weekly_spend_pre"] == 0)

# --- 캠페인 타입 x 기간 매핑 ---
day_to_week = tx[["DAY", "WEEK_NO"]].drop_duplicates().sort_values("DAY").reset_index(drop=True)
day_arr, week_arr = day_to_week["DAY"].values, day_to_week["WEEK_NO"].values
def day_to_week_lookup(day):
    idx = np.searchsorted(day_arr, day, side="right") - 1
    return week_arr[max(0, min(idx, len(day_arr) - 1))]
campaign_desc = campaign_desc.copy()
campaign_desc["START_WEEK"] = campaign_desc["START_DAY"].apply(day_to_week_lookup)
campaign_desc["END_WEEK"] = campaign_desc["END_DAY"].apply(day_to_week_lookup)
camp_full = campaign_table.merge(
    campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_WEEK", "END_WEEK"]],
    on="CAMPAIGN", how="left", suffixes=("", "_desc")
)
recipients = set(campaign_table["household_key"].unique())
never_recipients = set(all_households) - recipients

# --- 가구x주차 패널 ---
weeks_camp = list(range(CAMP_MIN_WEEK, CAMP_MAX_WEEK + 1))
panel_index = pd.MultiIndex.from_product([all_households, weeks_camp], names=["household_key", "WEEK_NO"])
panel = pd.DataFrame(index=panel_index).reset_index()
weekly_spend = (
    tx[(tx["WEEK_NO"] >= CAMP_MIN_WEEK) & (tx["WEEK_NO"] <= CAMP_MAX_WEEK)]
    .groupby(["household_key", "WEEK_NO"])["SALES_VALUE"].sum().rename("spend")
)
panel = panel.merge(weekly_spend, on=["household_key", "WEEK_NO"], how="left")
panel["spend"] = panel["spend"].fillna(0)

def build_active_weeks(camp_df, type_name):
    sub = camp_df[camp_df["DESCRIPTION_desc"] == type_name][["household_key", "START_WEEK", "END_WEEK"]].copy()
    sub["START_WEEK"] = sub["START_WEEK"].clip(lower=CAMP_MIN_WEEK)
    sub["END_WEEK"] = sub["END_WEEK"].clip(upper=CAMP_MAX_WEEK)
    sub["WEEK_NO"] = sub.apply(lambda r: list(range(int(r["START_WEEK"]), int(r["END_WEEK"]) + 1)), axis=1)
    sub = sub.explode("WEEK_NO")[["household_key", "WEEK_NO"]].drop_duplicates()
    sub["WEEK_NO"] = sub["WEEK_NO"].astype(int)
    sub[f"active_{type_name}"] = 1
    return sub

for t in ["TypeA", "TypeB", "TypeC"]:
    active = build_active_weeks(camp_full, t)
    panel = panel.merge(active, on=["household_key", "WEEK_NO"], how="left")
    panel[f"active_{t}"] = panel[f"active_{t}"].fillna(0).astype(int)

panel = panel.merge(tier_df[["tier3", "was_zero_pre"]], left_on="household_key", right_index=True, how="left")

# --- Type x 계층 상호작용 더미 ---
for t in ["TypeA", "TypeB", "TypeC"]:
    for tier in ["저지출", "중지출", "고지출"]:
        panel[f"{t}_x_{tier}"] = ((panel[f"active_{t}"] == 1) & (panel["tier3"] == tier)).astype(int)

interaction_cols = [f"{t}_x_{tier}" for t in ["TypeA", "TypeB", "TypeC"] for tier in ["저지출", "중지출", "고지출"]]

# --- 셀 크기표 (4단계) ---
def cell_size_table(tier_col):
    out = {}
    for t in ["TypeA", "TypeB", "TypeC"]:
        hh_active = panel.loc[panel[f"active_{t}"] == 1, ["household_key", tier_col]].drop_duplicates()
        out[t] = hh_active.groupby(tier_col, observed=True)["household_key"].nunique()
    return pd.DataFrame(out)
cell3 = cell_size_table("tier3")

# --- 6단계 본 회귀 ---
reg_df = panel.copy()
reg_df["entity"] = reg_df["household_key"]
reg_df["time"] = reg_df["WEEK_NO"]
reg_df = reg_df.set_index(["entity", "time"])

model = PanelOLS(
    dependent=reg_df["spend"],
    exog=reg_df[interaction_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
result = model.fit(cov_type="clustered", cluster_entity=True)

print("파이프라인 + 6단계 회귀 재실행 완료")
print(result.params)


파이프라인 + 6단계 회귀 재실행 완료
TypeA_x_저지출    1.056302
TypeA_x_중지출    0.819589
TypeA_x_고지출   -0.162506
TypeB_x_저지출    1.442018
TypeB_x_중지출    0.417724
TypeB_x_고지출   -2.429732
TypeC_x_저지출    9.675478
TypeC_x_중지출   -1.219311
TypeC_x_고지출   -6.252591
Name: parameter, dtype: float64


## 7단계 — 다중비교 보정 (Benjamini-Hochberg / FDR)

9개 조합의 p-value를 한꺼번에 검정했으므로, 개별 p<0.05만으로 유의성을 판단하지 않고
FDR(허위발견율) 5% 기준으로 보정.

In [4]:
coef_table = result.params.to_frame("coef")
coef_table["se"] = result.std_errors
coef_table["t"] = result.tstats
coef_table["p_raw"] = result.pvalues
coef_table = coef_table.reset_index().rename(columns={"index": "변수"})
coef_table[["Type", "계층"]] = coef_table["변수"].str.split("_x_", expand=True)

# 셀 크기 병합
cell_long = cell3.stack().reset_index()
cell_long.columns = ["계층", "Type", "수신가구수"]
coef_table = coef_table.merge(cell_long, on=["Type", "계층"], how="left")
coef_table["표본주의"] = coef_table["수신가구수"] < CELL_SIZE_THRESH

# --- BH(FDR) 보정 ---
reject, p_adj, _, _ = multipletests(coef_table["p_raw"], alpha=0.05, method="fdr_bh")
coef_table["p_adj_bh"] = p_adj
coef_table["유의_BH보정후"] = reject

pretrend_note = {"저지출": "5단계 통과(안전)", "중지출": "5단계 경계선(p=0.053)", "고지출": "5단계 검정력 부족"}
coef_table["평행추세_참고"] = coef_table["계층"].map(pretrend_note)

cols_order = ["Type", "계층", "coef", "se", "p_raw", "p_adj_bh", "유의_BH보정후", "수신가구수", "표본주의", "평행추세_참고"]
final_table = coef_table[cols_order].sort_values("p_raw")
print(final_table.to_string(index=False))


 Type  계층      coef       se    p_raw  p_adj_bh  유의_BH보정후  수신가구수  표본주의          평행추세_참고
TypeC 고지출 -6.252591 1.756420 0.000371  0.003341      True    295 False       5단계 검정력 부족
TypeB 고지출 -2.429732 0.965271 0.011832  0.053243     False    628 False       5단계 검정력 부족
TypeC 저지출  9.675478 6.939703 0.163253  0.489758     False     28  True       5단계 통과(안전)
TypeA 중지출  0.819589 0.687613 0.233289  0.524901     False    554 False 5단계 경계선(p=0.053)
TypeA 저지출  1.056302 1.431923 0.460710  0.805521     False    187 False       5단계 통과(안전)
TypeB 저지출  1.442018 2.335870 0.537014  0.805521     False     90 False       5단계 통과(안전)
TypeB 중지출  0.417724 1.362253 0.759116  0.833748     False    305 False 5단계 경계선(p=0.053)
TypeC 중지출 -1.219311 4.360564 0.779768  0.833748     False     74 False 5단계 경계선(p=0.053)
TypeA 고지출 -0.162506 0.774218 0.833748  0.833748     False    772 False       5단계 검정력 부족


In [5]:
print("="*60)
print("보정 전후 비교")
print("="*60)
n_sig_raw = (coef_table["p_raw"] < 0.05).sum()
n_sig_bh = coef_table["유의_BH보정후"].sum()
print(f"raw p<0.05 로 유의했던 조합: {n_sig_raw}개")
print(f"BH 보정 후에도 유의한 조합: {n_sig_bh}개")
print()
print("[BH 보정 후에도 살아남은 조합]")
survivors = final_table[final_table["유의_BH보정후"]]
print(survivors.to_string(index=False) if len(survivors) > 0 else "없음")
print()
print("[보정 전엔 유의했으나 보정 후 탈락한 조합]")
dropped = final_table[(final_table["p_raw"] < 0.05) & (~final_table["유의_BH보정후"])]
print(dropped.to_string(index=False) if len(dropped) > 0 else "없음")


보정 전후 비교
raw p<0.05 로 유의했던 조합: 2개
BH 보정 후에도 유의한 조합: 1개

[BH 보정 후에도 살아남은 조합]
 Type  계층      coef      se    p_raw  p_adj_bh  유의_BH보정후  수신가구수  표본주의    평행추세_참고
TypeC 고지출 -6.252591 1.75642 0.000371  0.003341      True    295 False 5단계 검정력 부족

[보정 전엔 유의했으나 보정 후 탈락한 조합]
 Type  계층      coef       se    p_raw  p_adj_bh  유의_BH보정후  수신가구수  표본주의    평행추세_참고
TypeB 고지출 -2.429732 0.965271 0.011832  0.053243     False    628 False 5단계 검정력 부족


## 결과 해석 가이드

**확인할 순서**

1. `유의_BH보정후 = True`인 조합만 "통계적으로 안정적인 신호"로 인정
2. 그중 `표본주의 = True`(저지출×TypeC, n=28)면 → 표본이 작아 계수가 불안정할 수 있으므로
   BH를 통과했더라도 신뢰도를 한 단계 낮춰서 서술
3. `평행추세_참고`가 "경계선"이나 "검정력 부족"인 계층에서 살아남은 조합은 →
   "캠페인 효과"와 "원래 있던 사전추세"를 완전히 분리하지 못했다는 단서를 반드시 병기
4. **고지출 계층에서 유의한 결과가 나왔다면 특히 주의** — 5단계에서 고지출 계층은
   미수신 가구가 49명뿐이라 평행추세 검정 자체의 힘이 약했던 계층임

## 다음 단계
여기서 최종 확정된 Type×계층 조합을 갖고 8단계(위약검정, 캠페인 미수신 916가구)로 진행.
위약검정에서 같은 패턴이 안 나타나야 비로소 "캠페인 효과일 가능성이 있다"고 말할 수 있음.